#### Checking the outputs

In [7]:
import google.generativeai as genai

# Configure API Key
genai.configure(api_key="API_Key")

# Create model with safety settings
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash-lite",

    safety_settings=[
        {
            "category": "HARM_CATEGORY_HARASSMENT",
            "threshold": "BLOCK_MEDIUM_AND_ABOVE"
        },
        { "category": "HARM_CATEGORY_HATE_SPEECH",
            "threshold": "BLOCK_MEDIUM_AND_ABOVE"
        },
        {
            "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
            "threshold": "BLOCK_MEDIUM_AND_ABOVE"
        },
        {
            "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
            "threshold": "BLOCK_MEDIUM_AND_ABOVE"
        }
    ]
)

In [24]:
def get_completion_from_messages(system_message, user_message):

    chat = model.start_chat(
        history=[
            {
                "role": "model",
                "parts": [system_message]
            }
        ]
    )
    response = chat.send_message(user_message, generation_config = {"temperature":0, "max_output_tokens" : 500})
    response.text
    return response

#### Check output for potentially harmful content

In [10]:
final_response_to_customer = f"""
The SmartX ProPhone has a 6.1-inch display, 128GB storage,
12MP dual camera, and 5G. The FotoSnap DSLR Camera
has a 24.2MP sensor, 1080p video, 3-inch LCD, and
interchangeable lenses. We have a variety of TVs, including
the CineView 4K TV with a 55-inch display, 4K resolution,
HDR, and smart TV features. We also have the SoundMax
Home Theater system with 5.1 channel, 1000W output, wireless
subwoofer, and Bluetooth. Do you have any specific questions
about these products or any other products we offer?
"""
response = model.generate_content(final_response_to_customer)

# Safety ratings
print(response.candidates[0].safety_ratings)

[]


#### Check if output is factually based on the provided p[roduct information

In [33]:
system_message = f"""
You are an assistant that evaluates whether \
customer service agent responses sufficiently \
answer customer questions, and also validates that \
all the facts assistant cites from the product \
The product information and user and customer service agent \
messages will be delimited by 3 backticks, ie.'''.
Respond with a Y or N character, with no punctuation:
Y- if the output sufficiently answer the question \
AND the response correctly uses product information
N - Otherwise

Output a single letter only.
"""
customer_message = f"""
tell me about the smartx pro phone and the fotosnap camera, the dslr one. \
"""
product_information = """{"name": "SmartX ProPhone", "category": "Smartphones and Accessories", 
"brand": "SmartX", "model_number": "SX-PP10", "warranty": "1 year", "rating": 4.6, "features": [ "6.1-inch display", "128GB storage", "12MP dual camera", "5G" ], 
"description": "A powerful smartphone with advanced camera features.", "price": 899.99 } { "name": "FotoSnap DSLR Camera", "category": "Cameras and Camcorders", "brand": "FotoSnap", 
"model_number": "FS-DSLR200", "warranty": "1 year", "rating": 4.7, "features": [ "24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses" ],
"description": "Capture stunning photos and videos with this versatile DSLR camera.", "price": 599.99 } { "name": "CineView 4K TV", "category": "Televisions and Home Theater Systems", 
"brand": "CineView", "model_number": "CV-4K55", "warranty": "2 years", "rating": 4.8, "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], "description": "A stunning 4K TV with vibrant colors and smart features.", "price": 599.99 } { "name": "SoundMax Home Theater", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", 
"model_number": "SM-HT100", "warranty": "1 year", "rating": 4.4, "features": [ "5.1 channel", "1000W output", "Wireless subwoofer", "Bluetooth" ], "description": "A powerful home theater system for an immersive audio experience.", "price": 399.99 } { "name": "CineView 8K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-8K65", "warranty": "2 years", 
"rating": 4.9, "features": [ "65-inch display", "8K resolution", "HDR", "Smart TV" ], "description": "Experience the future of television with this stunning 8K TV.", "price": 2999.99 } { "name": "SoundMax Soundbar", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-SB50", "warranty": "1 year", "rating": 4.3, "features": [ "2.1 channel", "300W output", "Wireless subwoofer", 
"Bluetooth" ], "description": "Upgrade your TV's audio with this sleek and powerful soundbar.", "price": 199.99 } { "name": "CineView OLED TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", "model_number": "CV-OLED55", "warranty": "2 years", "rating": 4.7, "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], "description": "Experience true blacks and vibrant colors with this OLED TV.", "price": 1499.99 }"""

q_a_pair = f"""
Customer message : '''{customer_message}'''
Product information : '''{product_information}'''
Agent response : '''{final_response_to_customer}'''

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question

Output Y or N
"""

# Generate response
response = get_completion_from_messages(
    system_message,
    q_a_pair
)

print(response.text)

Y


In [30]:
another_response = "Like is like a box of chocolates"
q_a_pair = f"""
Customer message : '''{customer_message}'''
Product information : '''{product_information}'''
Agent response : '''{another_response}

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question

Output Y or N
"""

response = get_completion_from_messages(system_message, q_a_pair)
print(response.text)

N
